# 📈 Stock Price Prediction using LSTM
This notebook demonstrates how to use Long Short-Term Memory (LSTM) networks for predicting stock prices.
- Uses Yahoo Finance
- Preprocessing with MinMaxScaler
- Builds and trains LSTM model
- Optional Gradio interface for interaction

In [ ]:
!pip install yfinance gradio matplotlib scikit-learn tensorflow

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import gradio as gr

In [ ]:
def get_stock_data(ticker='AAPL', start='2015-01-01', end='2023-12-31'):
    df = yf.download(ticker, start=start, end=end)
    return df[['Close']]

df = get_stock_data()
df.tail()

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df)
train_len = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_len]
test_data = scaled_data[train_len - 60:]

def create_sequences(data, seq_length=60):
    x, y = [], []
    for i in range(seq_length, len(data)):
        x.append(data[i-seq_length:i, 0])
        y.append(data[i, 0])
    return np.array(x), np.array(y)

x_train, y_train = create_sequences(train_data)
x_test, y_test = create_sequences(test_data)

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

In [ ]:
model = Sequential([
    LSTM(units=50, return_sequences=True, input_shape=(x_train.shape[1], 1)),
    LSTM(units=50),
    Dense(1)
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(x_train, y_train, epochs=5, batch_size=32)

In [ ]:
predictions = model.predict(x_test)
predictions = scaler.inverse_transform(predictions.reshape(-1, 1))
actual = scaler.inverse_transform(y_test.reshape(-1, 1))

plt.figure(figsize=(12, 6))
plt.plot(actual, label='Actual')
plt.plot(predictions, label='Predicted')
plt.legend()
plt.title('Stock Price Prediction')
plt.show()

In [ ]:
def predict_stock(symbol):
    df = get_stock_data(symbol)
    scaled_data = scaler.fit_transform(df)
    test_data = scaled_data[-120:]
    x, _ = create_sequences(test_data)
    x = x.reshape((x.shape[0], x.shape[1], 1))
    prediction = model.predict(x)
    final_pred = scaler.inverse_transform(prediction)
    return f"Predicted latest close price for {symbol}: ${final_pred[-1][0]:.2f}"

gr.Interface(fn=predict_stock, inputs='text', outputs='text', title='📈 Stock Price Predictor').launch()